## Step 3A: Generate new molecules and reaction pathways with DORAnet

This step expands each starter molecule into an enzymatic reaction network with
`DORAnet`, enumerates the molecules reachable from that starter, and finds
biosynthetic pathways to each of them. It runs from the command line.

### Directory layout

All generation runs live under `Step3A_Antiviral_Compounds/`, one subdirectory
per starter molecule:

```
Step3A_Antiviral_Compounds/
├── Adenosine_Starter/
├── Brincidofovir_Starter/
├── Guanosine_Starter/
├── Inosine_Starter/
├── Remdesivir_Starter/
└── Xanthosine_Starter/
```

Each starter directory contains the DORAnet scripts and its own config:

```
Inosine_Starter/
├── runDORAnet.py         # script for DORAnet
├── doranet_config.yaml   # the only file you change per starter
└── ...                   # saves reaction network, and pathway files after a run
```

`doranet_config.yaml` sets the DORAnet checkout path (`doranetPath`), the output
prefix (`jobName`), the `starters` SMILES, the network settings (`generations`,
`direction`, `ruleset`, `maxAtoms`), the pathway depth (`totalGenerations`), and
an optional `helpers` set. To add a new starter, copy an existing `*_Starter`
directory and change `starters` and `jobName` in its config; nothing else needs
editing.

### How to run

Generate for a single starter:

```bash
cd Step3A_Antiviral_Compounds/Inosine_Starter
python runDORAnet.py doranet_config.yaml
```

Generate for every starter in one pass:

```bash
for d in Step3A_Antiviral_Compounds/*_Starter/ ; do
    ( cd "$d" && python runDORAnet.py doranet_config.yaml )
done
```

For large networks, submit each starter to the HPC scheduler instead of running
interactively, since generation and pathway search can be memory and
time-intensive.

### Prerequisites

- A local DORAnet checkout at the `doranetPath` set in each config; the runner
  adds it to `sys.path` at runtime.
- An active environment that includes DORAnet, RDKit, and PyYAML (
  `SynThera_ExptDesigns`).
- Runtime scales with network size: every generated molecule
  except the starters and helpers becomes a pathway target, so tune
  `generations` and `maxAtoms` to keep runs tractable.

### Outputs and next step

Each starter directory holds its own outputs, prefixed with that run's `jobName`:

- `{jobName}_molecules.csv` — every generated molecule with its formula, exact
  mass, and heavy-atom count, flagged by whether it is a starter.
- Network and pathway files with the same `{jobName}` prefix, describing the
  biosynthetic routes to each target.

These generated molecules feed the next stage, where novel drug-like compounds
are filtered from the retro-biosynthesis output and then scored for potency and
ADMET properties.

# Step 3B: Aggregate DORAnet-generated antivirals across all starters

This notebook post-processes the DORAnet runs from Step 3A. For every
`*_Starter` directory it traces which molecules are derived from that starter,
filters out cofactors and non-drug-like products, and separates the
high-feasibility subset using the precomputed DORA-XGB (rule3) reaction scores.

For each starter it writes three per-starter files to `resultsDir`
(`{starter}_reactionDF.csv`, `{starter}_generated_compounds.csv`,
`{starter}_generated_compounds_highFeasibility.csv`), then aggregates every
starter into two master tables, also in `resultsDir`:

- **All possible target compounds** — one row per (starter, derived compound):
  `Starter_Name`, `Starter_Canonical_SMILES`, `Target_Canonical_SMILES`, `DORAnet_gen`.
- **High-feasibility target compounds** — the same columns plus
  `feasibilityScore_rule3`.

The pathogen and paths come from the standard job-configuration block, so the
notebook runs for any target by changing `target`.

In [1]:
import os
import io
import re
import math
import json
import glob
import yaml
import base64
from io import BytesIO
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, Draw, QED, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker

from PIL import Image, ImageDraw, ImageFont
from IPython.display import HTML, display, SVG

RDLogger.DisableLog('rdApp.*')

from DORA_XGB import DORA_XGB
by_desc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_descending_MW')
by_asc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_ascending_MW')
add_concat_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_concat')
add_subtract_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_subtract')

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None

<CONDA_ENV>/lib/python3.10/site-packages/xgboost/compat.py:105: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Configuration

In [2]:
# Anchor the project root explicitly so paths do not depend on the launch directory.
projectRoot = os.path.expanduser("<PROJECT_ROOT>")

# Set target to any pathogen present in your data.
target = "Ebola"

# Results location, derived from the project root. All outputs land here.
resultsDir = os.path.join(projectRoot, target, f"Results_{target}")
os.makedirs(resultsDir, exist_ok=True)

# Parent directory that holds the per-starter DORAnet results
starterParentDir = os.path.join(resultsDir, "Step3A_Antiviral_Compounds")

# DORAnet cofactor table (used to exclude cofactors and endogenous metabolites).
doranetCofactorsPath = os.path.join(projectRoot, "Input_Data", "doranet_all_cofactors.tsv")

# Helper / small molecules to exclude from the derived compound set.
helpersToExclude = {
    "O", "O=O", "[H][H]", "O=C=O", "C=O", "[C-]#[O+]", "Br", "[Br][Br]",
    "CO", "C=C", "O=S(O)O", "N", "O=S(=O)(O)O", "O=NO", "N#N",
    "O=[N+]([O-])O", "NO", "C#N", "S", "O=S=O", "N#CO", "[H+]", "OO",
    "Cl", "I", "O=C(O)O", "O=P(O)(O)O", "O=P(O)(O)OP(=O)(O)O", "C",
    "CC", "CC=O", "CC(=O)O", "CCC(=O)O",
}

# Fingerprint, drug-likeness, and cofactor/endogenous cutoffs.
fpRadius, fpBits = 2, 2048
lipinskiMaxViolations = 0
physicochemicalCutoffs = {"mwMax": 700.0, "tpsaMax": 200.0, "hbdMax": 7, "qedMin": 0.10}

# DORA-XGB feasibility settings. rule3 (add_concat) is the published model.
feasibilityLabelCol = "feasibilityLabel_rule3"
feasibilityScoreCol = "feasibilityScore_rule3"
feasibilityTargetValue = 1
feasibilityFileName = "reactionDF_wDORAXGBfeasibility.csv"

# Fallback network depth if a starter doranet_config.yaml lacks 'generations'.
defaultGenerations = 3

# Tables used only to label pathway figures (optional; degrade gracefully if absent).
cofactorTablePathForPlots = doranetCofactorsPath
rulesetPathForPlots = os.path.join(projectRoot, "Input_Data", "JN3604IMT_rules.tsv")

# Load the DORAnet cofactor set once (starter-independent).
_cofactorTable = pd.read_csv(doranetCofactorsPath, sep="\t")
_cofactorTable.columns = [col.lstrip("#").strip() for col in _cofactorTable.columns]
cofactorExclusionSet = {
    Chem.MolToSmiles(mol)
    for smi in _cofactorTable["SMILES"].dropna()
    if (mol := Chem.MolFromSmiles(str(smi))) is not None
}

print("Target:", target)
print("Results directory:", os.path.abspath(resultsDir))
print("Starter parent directory:", os.path.abspath(starterParentDir))
print("Cofactors loaded from:", os.path.abspath(doranetCofactorsPath))
print("Cofactors loaded:", len(cofactorExclusionSet))

Target: Ebola
Results directory: <PROJECT_ROOT>/Ebola/Results_Ebola
Starter parent directory: <PROJECT_ROOT>/Ebola/Results_Ebola/Step3A_Antiviral_Compounds
Cofactors loaded from: <PROJECT_ROOT>/Input_Data/doranet_all_cofactors.tsv
Cofactors loaded: 43


### Helper functions

In [3]:
def splitMoleculeString(moleculeString):
    if moleculeString is None or (isinstance(moleculeString, float) and pd.isna(moleculeString)):
        return []
    return [mol for mol in str(moleculeString).split(".") if mol]


def readReactionDf(jsonPath):
    """Parse a DORAnet *_network_pretreated.json into a reaction DataFrame."""
    with open(jsonPath, "r", encoding="utf-8") as handle:
        reactionList = json.load(handle)

    records = []
    for rxn in reactionList:
        parts = str(rxn).split(">")
        if len(parts) != 4:
            continue
        reactants, ruleName, metaBlock, products = parts
        metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
        thermo, reactantStoich, productStoich, reactionType = metaParts
        records.append({
            "reactants": reactants.strip(),
            "products": products.strip(),
            "reactionString": f"{reactants.strip()} >> {products.strip()}",
            "ruleName": ruleName.strip(),
            "thermo": thermo,
            "reactionType": reactionType,
        })
    return pd.DataFrame(records)


def morganFp(mol):
    return rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, fpRadius, nBits=fpBits) if mol is not None else None


def isCofactorOrEndogenous(mol, canonSmi, cofactorExclusionSet, cutoffs):
    if mol is None:
        return True
    if canonSmi in cofactorExclusionSet:
        return True
    if (Descriptors.MolWt(mol) > cutoffs["mwMax"]
            or rdMolDescriptors.CalcTPSA(mol) > cutoffs["tpsaMax"]
            or rdMolDescriptors.CalcNumHBD(mol) > cutoffs["hbdMax"]):
        return True
    if QED.qed(mol) < cutoffs["qedMin"]:
        return True
    return False


def lipinskiViolationCount(mol):
    if mol is None:
        return None
    violations = 0
    if Descriptors.MolWt(mol) > 500:
        violations += 1
    if Descriptors.MolLogP(mol) > 5:
        violations += 1
    if rdMolDescriptors.CalcNumHBD(mol) > 5:
        violations += 1
    if rdMolDescriptors.CalcNumHBA(mol) > 10:
        violations += 1
    return violations


def passesLipinski(canonSmi, maxViolations):
    violations = lipinskiViolationCount(Chem.MolFromSmiles(str(canonSmi)))
    return violations is not None and violations <= maxViolations


def nearestStarterBySimilarity(mol, starterFpByCanon):
    productFp = morganFp(mol)
    if productFp is None or not starterFpByCanon:
        return None, -1.0
    bestStarter, bestSimilarity = None, -1.0
    for starterCanon, starterFp in starterFpByCanon.items():
        similarity = DataStructs.TanimotoSimilarity(productFp, starterFp)
        if similarity > bestSimilarity:
            bestSimilarity, bestStarter = similarity, starterCanon
    return bestStarter, bestSimilarity


def buildStarterDerivedSet(reactionDF, starterMols, canonStarterSet, canonHelperSet,
                           cofactorExclusionSet, cutoffs, starterFpByCanon):
    molToReactionIdxs = defaultdict(list)
    for idx, row in reactionDF.iterrows():
        for smi in splitMoleculeString(row["reactants"]):
            molToReactionIdxs[canonicalizeSmiles(smi)].append(idx)

    starterDerivedSet = {canonSmi: (canonSmi, 0, 1.0, None) for canonSmi, _ in starterMols}
    queue = deque((canonSmi, 0) for canonSmi in canonStarterSet)
    visited = set(canonStarterSet)

    while queue:
        currentCanon, currentDepth = queue.popleft()
        if currentCanon in canonHelperSet:
            continue

        for rxnIdx in molToReactionIdxs.get(currentCanon, []):
            row = reactionDF.loc[rxnIdx]
            productScores = []
            for productSmi in splitMoleculeString(row["products"]):
                productCanon = canonicalizeSmiles(productSmi)
                if productCanon in canonHelperSet or productCanon in starterDerivedSet:
                    continue
                productMol = Chem.MolFromSmiles(str(productSmi))
                if isCofactorOrEndogenous(productMol, productCanon, cofactorExclusionSet, cutoffs):
                    continue
                matchedStarter, similarity = nearestStarterBySimilarity(productMol, starterFpByCanon)
                if matchedStarter is None:
                    continue
                productScores.append((similarity, productCanon, matchedStarter))

            if not productScores:
                continue

            productScores.sort(key=lambda t: t[0], reverse=True)
            keptSimilarity, bestProductCanon, bestMatchedStarter = productScores[0]
            runnerUpSimilarity = productScores[1][0] if len(productScores) > 1 else None

            if bestProductCanon not in visited:
                visited.add(bestProductCanon)
                starterDerivedSet[bestProductCanon] = (
                    bestMatchedStarter, currentDepth + 1, keptSimilarity, runnerUpSimilarity
                )
                queue.append((bestProductCanon, currentDepth + 1))

    return starterDerivedSet


def enumerateStarterDerivedProducts(uniqueProductList, starterDerivedSet, canonHelperSet,
                                    canonStarterSet, cofactorExclusionSet, cutoffs):
    records = []
    for smi in uniqueProductList:
        mol = Chem.MolFromSmiles(str(smi))
        canonSmi = Chem.MolToSmiles(mol) if mol else None
        if canonSmi is None or canonSmi in canonHelperSet or canonSmi in canonStarterSet:
            continue
        if isCofactorOrEndogenous(mol, canonSmi, cofactorExclusionSet, cutoffs):
            continue
        if canonSmi not in starterDerivedSet:
            continue
        matchedRaw, generationDepth, keptSim, runnerUp = starterDerivedSet.get(
            canonSmi, (None, None, None, None)
        )
        records.append({
            "starter_Canonical_SMILES": canonicalizeSmiles(matchedRaw) if matchedRaw else None,
            "Canonical_SMILES": canonSmi,
            "generationSteps": generationDepth,
            "DerivativeSimScore": round(keptSim, 4) if keptSim is not None else None,
            "NonDerivativeSimScore": round(runnerUp, 4) if runnerUp is not None else None,
            "_molecularWeight": round(Descriptors.MolWt(mol), 4),
        })
    return pd.DataFrame(records)


def ensureReactantProductCols(df):
    df = df.copy()
    if "reactants" in df.columns and "products" in df.columns:
        return df.reset_index(drop=True)
    if "reactionString" in df.columns:
        sides = df["reactionString"].astype(str).str.split(">>", n=1, expand=True)
        df["reactants"] = sides[0].str.replace(" ", "", regex=False)
        df["products"] = sides[1].str.replace(" ", "", regex=False)
        return df.reset_index(drop=True)
    raise KeyError("Feasibility table needs 'reactants'/'products' or 'reactionString' columns")


def dedupLipinskiSort(rawDF):
    """Deduplicate on canonical SMILES, keep Lipinski-compliant rows, sort by MW."""
    if rawDF.empty:
        return rawDF
    dedup = rawDF.drop_duplicates(subset="Canonical_SMILES", keep="first").reset_index(drop=True)
    mask = dedup["Canonical_SMILES"].apply(lambda s: passesLipinski(s, lipinskiMaxViolations))
    kept = dedup[mask].reset_index(drop=True)
    return (
        kept.sort_values("_molecularWeight", ascending=False)
        .drop(columns="_molecularWeight")
        .reset_index(drop=True)
    )


def buildFeasibilityScoreByCompound(reactionDfHighFeasible, scoreCol):
    """Assign each compound the maximum rule3 score among high-feasibility
    reactions that produce it."""
    scoreByCompound = {}
    if scoreCol not in reactionDfHighFeasible.columns:
        return scoreByCompound
    for _, row in reactionDfHighFeasible.iterrows():
        try:
            score = float(row[scoreCol])
        except (TypeError, ValueError):
            continue
        for productSmi in splitMoleculeString(row["products"]):
            canon = canonicalizeSmiles(productSmi)
            if canon is None:
                continue
            if canon not in scoreByCompound or score > scoreByCompound[canon]:
                scoreByCompound[canon] = score
    return scoreByCompound


def readGenerationsFromConfig(starterDir, default):
    """Read the DORAnet network depth (generations) from the starter's config."""
    configPath = os.path.join(starterDir, "doranet_config.yaml")
    if not os.path.isfile(configPath):
        print(f"[warn] {os.path.basename(starterDir)}: no doranet_config.yaml; using generations={default}")
        return default
    with open(configPath) as handle:
        cfg = yaml.safe_load(handle) or {}
    return int(cfg.get("generations", default))


### Process one starter directory

`processStarter` reads a single `*_Starter` folder's DORAnet outputs and returns
the all possible and high feasibility compounds for that starter. 

In [4]:
def toMasterSchema(richDF, starterName, withFeasibility=False):
    """Map a rich per-starter frame to the master output columns."""
    baseCols = ["Starter_Name", "Starter_Canonical_SMILES", "Target_Canonical_SMILES", "DORAnet_gen"]
    if richDF.empty:
        return pd.DataFrame(columns=baseCols + ([feasibilityScoreCol] if withFeasibility else []))
    out = pd.DataFrame({
        "Starter_Name": starterName,
        "Starter_Canonical_SMILES": richDF["starter_Canonical_SMILES"],
        "Target_Canonical_SMILES": richDF["Canonical_SMILES"],
        "DORAnet_gen": richDF["generationSteps"],
    })
    if withFeasibility:
        out[feasibilityScoreCol] = richDF[feasibilityScoreCol]
    return out


def processStarter(starterDir, starterName):
    jsonMatches = sorted(glob.glob(os.path.join(starterDir, "*_network_pretreated.json")))
    molMatches = sorted(glob.glob(os.path.join(starterDir, "*_molecules.csv")))
    if not jsonMatches or not molMatches:
        print(f"[skip] {starterName}: missing network json or molecules csv")
        return None, None

    # Starter SMILES from the molecules CSV(s).
    userStarters = set()
    for molCsv in molMatches:
        tempDF = pd.read_csv(molCsv, usecols=["SMILES", "Is_Starter"])
        userStarters.update(tempDF[tempDF["Is_Starter"] == True]["SMILES"].dropna().tolist())
    if not userStarters:
        print(f"[skip] {starterName}: no starter rows found")
        return None, None

    reactionDF = readReactionDf(jsonMatches[0])

    # Network depth for this starter, read from its doranet_config.yaml.
    generations = readGenerationsFromConfig(starterDir, defaultGenerations)

    # Save the parsed reactions for this starter (reaction pathways).
    reactionPath = os.path.join(resultsDir, f"{starterName}_reaction_pathways.csv")
    reactionDF.to_csv(reactionPath, index=False)

    # Per-starter sets.
    starterMols = [
        (canonicalizeSmiles(smi), Chem.MolFromSmiles(str(smi)))
        for smi in userStarters if Chem.MolFromSmiles(str(smi)) is not None
    ]
    canonStarterSet = {s for s, _ in starterMols}
    canonHelperSet = {canonicalizeSmiles(s) for s in helpersToExclude}
    starterFpByCanon = {
        s: morganFp(mol) for s, mol in starterMols if morganFp(mol) is not None
    }

    # All-possible: BFS over the full network.
    starterDerivedSet = buildStarterDerivedSet(
        reactionDF, starterMols, canonStarterSet, canonHelperSet,
        cofactorExclusionSet, physicochemicalCutoffs, starterFpByCanon,
    )
    allProducts = reactionDF["products"].apply(splitMoleculeString).explode().dropna().unique().tolist()
    allRaw = enumerateStarterDerivedProducts(
        allProducts, starterDerivedSet, canonHelperSet, canonStarterSet,
        cofactorExclusionSet, physicochemicalCutoffs,
    )
    generatedCompoundsDF = dedupLipinskiSort(allRaw)

    # Save the full per-starter generated compounds (rich columns).
    generatedPath = os.path.join(resultsDir, f"{starterName}_generated_compounds.csv")
    generatedCompoundsDF.to_csv(generatedPath, index=False)

    allOut = toMasterSchema(generatedCompoundsDF, starterName, withFeasibility=False)

    # High-feasibility: requires the DORA-XGB feasibility table.
    feasPath = os.path.join(starterDir, feasibilityFileName)
    if not os.path.isfile(feasPath):
        print(f"[warn] {starterName}: no {feasibilityFileName}; high-feasibility table skipped")
        hfOut = pd.DataFrame(columns=list(allOut.columns) + [feasibilityScoreCol])
        starterContexts[starterName] = {
            "generations": generations,
            "canonStarterSet": canonStarterSet,
            "canonHelperSet": canonHelperSet,
            "reactionDF_highFeasible": None,
            "generatedCompoundsDF_highFeasibility": None,
        }
        print(f"[done] {starterName}: allPossible={len(allOut)}, highFeasibility=0")
        return allOut, hfOut

    reactionDF_DORAXGB = pd.read_csv(feasPath)
    reactionDF_highFeasible = ensureReactantProductCols(
        reactionDF_DORAXGB[
            (reactionDF_DORAXGB[feasibilityLabelCol] == feasibilityTargetValue)
            & (reactionDF_DORAXGB[feasibilityLabelCol].notna())
        ]
    )

    starterDerivedSetHF = buildStarterDerivedSet(
        reactionDF_highFeasible, starterMols, canonStarterSet, canonHelperSet,
        cofactorExclusionSet, physicochemicalCutoffs, starterFpByCanon,
    )
    hfProducts = reactionDF_highFeasible["products"].apply(splitMoleculeString).explode().dropna().unique().tolist()
    hfRaw = enumerateStarterDerivedProducts(
        hfProducts, starterDerivedSetHF, canonHelperSet, canonStarterSet,
        cofactorExclusionSet, physicochemicalCutoffs,
    )
    generatedCompoundsDF_highFeasibility = dedupLipinskiSort(hfRaw)

    # Preserve the full-network generation depth rather than the sparser subnetwork's.
    stepsByCompound = generatedCompoundsDF.drop_duplicates("Canonical_SMILES").set_index("Canonical_SMILES")["generationSteps"]
    if not generatedCompoundsDF_highFeasibility.empty:
        generatedCompoundsDF_highFeasibility["generationSteps"] = (
            generatedCompoundsDF_highFeasibility["Canonical_SMILES"].map(stepsByCompound)
        )

    # Per-compound feasibility score: max rule3 score among producing reactions.
    scoreByCompound = buildFeasibilityScoreByCompound(reactionDF_highFeasible, feasibilityScoreCol)
    if not generatedCompoundsDF_highFeasibility.empty:
        generatedCompoundsDF_highFeasibility[feasibilityScoreCol] = (
            generatedCompoundsDF_highFeasibility["Canonical_SMILES"].map(scoreByCompound)
        )

    # Save the per-starter high-feasibility compounds (rich columns + feasibility score).
    highFeasPath = os.path.join(resultsDir, f"{starterName}_generated_compounds_highFeasibility.csv")
    generatedCompoundsDF_highFeasibility.to_csv(highFeasPath, index=False)

    hfOut = toMasterSchema(generatedCompoundsDF_highFeasibility, starterName, withFeasibility=True)

    # Retain everything the pathway-plotting cells need for this starter.
    starterContexts[starterName] = {
        "generations": generations,
        "canonStarterSet": canonStarterSet,
        "canonHelperSet": canonHelperSet,
        "reactionDF_highFeasible": reactionDF_highFeasible,
        "generatedCompoundsDF_highFeasibility": generatedCompoundsDF_highFeasibility,
    }

    print(f"[done] {starterName}: allPossible={len(allOut)}, highFeasibility={len(hfOut)}")
    return allOut, hfOut

## Aggregate all starters and save the master tables

Every `*_Starter` directory under `starterParentDir` is processed and the results
are concatenated into two master tables written to `resultsDir`.

In [6]:
starterDirs = sorted(glob.glob(os.path.join(starterParentDir, "*_Starter")))
print(f"Found {len(starterDirs)} starter directories:")
for d in starterDirs:
    print("  ", os.path.basename(d))

starterContexts = {}
allFrames, hfFrames = [], []
for starterDir in starterDirs:
    starterName = os.path.basename(starterDir).replace("_Starter", "")
    allOut, hfOut = processStarter(starterDir, starterName)
    if allOut is not None and not allOut.empty:
        allFrames.append(allOut)
    if hfOut is not None and not hfOut.empty:
        hfFrames.append(hfOut)

outColumns = ["Starter_Name", "Starter_Canonical_SMILES", "Target_Canonical_SMILES", "DORAnet_gen"]

masterAllPossible = (
    pd.concat(allFrames, ignore_index=True)
    .drop_duplicates(subset=["Starter_Name", "Target_Canonical_SMILES"])
    .reset_index(drop=True)
    if allFrames else pd.DataFrame(columns=outColumns)
)
masterHighFeasibility = (
    pd.concat(hfFrames, ignore_index=True)
    .drop_duplicates(subset=["Starter_Name", "Target_Canonical_SMILES"])
    .reset_index(drop=True)
    if hfFrames else pd.DataFrame(columns=outColumns + [feasibilityScoreCol])
)

allPath = os.path.join(resultsDir, f"{target}_antivirals_allPossible.csv")
hfPath = os.path.join(resultsDir, f"{target}_antivirals_highFeasibility.csv")
masterAllPossible.to_csv(allPath, index=False)
masterHighFeasibility.to_csv(hfPath, index=False)

print(f"\nAll possible target compounds : {len(masterAllPossible):,} rows -> {allPath}")
print(f"High-feasibility target compounds: {len(masterHighFeasibility):,} rows -> {hfPath}")
print("\nPer-starter counts (all possible):")
if not masterAllPossible.empty:
    print(masterAllPossible["Starter_Name"].value_counts().to_string())

Found 6 starter directories:
   Adenosine_Starter
   Brincidofovir_Starter
   Guanosine_Starter
   Inosine_Starter
   Remdesivir_Starter
   Xanthosine_Starter
[done] Adenosine: allPossible=4045, highFeasibility=121
[done] Brincidofovir: allPossible=2932, highFeasibility=1
[done] Guanosine: allPossible=3962, highFeasibility=346
[done] Inosine: allPossible=4122, highFeasibility=197
[done] Remdesivir: allPossible=618, highFeasibility=0
[done] Xanthosine: allPossible=3398, highFeasibility=121

All possible target compounds : 19,077 rows -> <PROJECT_ROOT>/Ebola/Results_Ebola/Ebola_antivirals_allPossible.csv
High-feasibility target compounds: 786 rows -> <PROJECT_ROOT>/Ebola/Results_Ebola/Ebola_antivirals_highFeasibility.csv

Per-starter counts (all possible):
Starter_Name
Inosine          4122
Adenosine        4045
Guanosine        3962
Xanthosine       3398
Brincidofovir    2932
Remdesivir        618


## Plot pathways

The pathway renderers below reproduce the original notebook. They operate on one
starter at a time, so choose a starter here; the cell wires up the exact
variables the plotting cells expect, using that starter's own network depth
(`generations`) read from its `doranet_config.yaml`.

In [7]:
# Choose which starter's pathways to plot. Change this to any discovered starter.
starterToPlot = "Inosine"

if starterToPlot not in starterContexts:
    raise KeyError(f"'{starterToPlot}' not found. Available: {sorted(starterContexts)}")

_ctx = starterContexts[starterToPlot]
if _ctx["reactionDF_highFeasible"] is None:
    raise ValueError(
        f"'{starterToPlot}' has no high-feasibility data (missing {feasibilityFileName}); cannot plot."
    )

# Variables the original plotting cells expect.
reactionDF_highFeasible = _ctx["reactionDF_highFeasible"]
reactionDF_DORAXGB_highFeasibility = _ctx["reactionDF_highFeasible"]
generatedCompoundsDF_highFeasibility = _ctx["generatedCompoundsDF_highFeasibility"]
canonStarterSet = _ctx["canonStarterSet"]
canonHelperSet = _ctx["canonHelperSet"]
doranet_generations = _ctx["generations"]
bestRuleScoreCol = feasibilityScoreCol
fileNamePrefix = starterToPlot

print(f"Plotting pathways for {starterToPlot} (network depth {doranet_generations})")
print(f"  high-feasibility reactions: {len(reactionDF_highFeasible):,}")
print(f"  high-feasibility compounds: {len(generatedCompoundsDF_highFeasibility):,}")

Plotting pathways for Inosine (network depth 3)
  high-feasibility reactions: 1,324
  high-feasibility compounds: 197


### Feasible pathways (vertical, per reaction step)

In [ ]:
# CONFIG
numToDisplayPerStep = 4
saveFigures         = False
figureOutDir        = os.path.join(resultsDir, "PathwayFigures")
subImgSize          = (600, 300)
articleFiguresDir     = os.path.join(resultsDir, "PathwayFigures")
rulesetPath           = rulesetPathForPlots
cofactorTablePath     = cofactorTablePathForPlots

cofactorDisplayOverride = {
    "CO-A": "CoA", "ACETYL-COA": "Acetyl-CoA", "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O", "HYDROGEN-PEROXIDE": "H2O2", "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3", "S-ADENOSYLMETHIONINE": "SAM", "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate", "GLT": "Glutamate", "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

separatorFont = loadFont(52)
labelFont     = loadFont(18)
headerFont    = loadFont(22)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactor names will not be applied.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def cofactorNameFor(smi):
    return cofactorNameByKey.get(cofactorKey(smi))

def rdkitImageToPil(imageObj):
    if isinstance(imageObj, Image.Image):
        return imageObj.convert("RGB")
    if isinstance(imageObj, (bytes, bytearray)):
        return Image.open(BytesIO(imageObj)).convert("RGB")
    return Image.open(BytesIO(imageObj.data)).convert("RGB")

def centeredText(draw, text, imgWidth, y, font, color="black"):
    try:
        bbox  = draw.textbbox((0, 0), text, font=font)
        textW = bbox[2] - bbox[0]
    except Exception:
        textW = len(text) * 8
    draw.text(((imgWidth - textW) // 2, y), text, fill=color, font=font)

def makeTextBanner(text, bannerWidth, bannerHeight=40, bgColor="white", font=None):
    img = Image.new("RGB", (bannerWidth, bannerHeight), bgColor)
    centeredText(ImageDraw.Draw(img), text, bannerWidth, (bannerHeight - 24) // 2, font or headerFont)
    return img

def stackPanelsVertically(panelList, gapSize=14, bgColor="white"):
    valid = [p for p in panelList if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid) + gapSize * (len(valid) - 1)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height + gapSize
    return canvas

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

def reorderStepsForward(steps, canonStarterSet):
    if len(steps) <= 1:
        return steps
    parsed = []
    for smi in steps:
        if ">>" not in str(smi):
            continue
        rStr, pStr = str(smi).split(">>", 1)
        parsed.append({
            "smi"      : smi,
            "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
            "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
        })
    if not parsed:
        return steps
    firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
    if firstIdx is None:
        return steps
    ordered   = [parsed[firstIdx]]
    remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["products"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return [s["smi"] for s in ordered]

def getMolLabel(smi, canonStarterSet, canonTargetSmi, side="reactant"):
    canon = canonicalizeSmiles(smi)
    if canon in canonStarterSet:
        return "Starter compound"
    if canon == canonTargetSmi:
        return "Target compound"
    cofactorName = cofactorNameFor(smi)
    if cofactorName:
        return cofactorName
    return "Reactant" if side == "reactant" else "Product"

def makeLabeledMolImage(smi, label, subImageSize=(280, 210)):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    molImg      = rdkitImageToPil(Draw.MolToImage(mol, size=subImageSize))
    labelBanner = Image.new("RGB", (molImg.width, 34), "#f0f0f0")
    centeredText(ImageDraw.Draw(labelBanner), label, molImg.width, 7, labelFont, "#333333")
    combined = Image.new("RGB", (molImg.width, molImg.height + labelBanner.height), "white")
    combined.paste(molImg,      (0, 0))
    combined.paste(labelBanner, (0, molImg.height))
    return combined

def makeSeparatorImage(text, height, width=90):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        bbox         = draw.textbbox((0, 0), text, font=separatorFont)
        textW, textH = bbox[2] - bbox[0], bbox[3] - bbox[1]
    except Exception:
        textW, textH = 30, 50
    draw.text(((width - textW) // 2, (height - textH) // 2), text, fill="#222222", font=separatorFont)
    return img

def makeCustomReactionPanel(rxnSmi, canonStarterSet, canonTargetSmi,
                             panelTitle, subImageSize=(280, 210)):
    if ">>" not in str(rxnSmi):
        return None
    reactantStr, productStr = str(rxnSmi).split(">>", 1)
    reactantSmiles = (
        [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) in canonStarterSet]
        + [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) not in canonStarterSet]
    )
    productSmiles = (
        [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) != canonTargetSmi]
        + [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) == canonTargetSmi]
    )
    reactantImgs = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="reactant"), subImageSize)
        for s in reactantSmiles] if img is not None]
    productImgs  = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="product"), subImageSize)
        for s in productSmiles]  if img is not None]
    if not reactantImgs or not productImgs:
        return None
    maxH  = max(img.height for img in reactantImgs + productImgs)
    gap   = 16
    parts = []
    for i, img in enumerate(reactantImgs):
        parts.append(img)
        if i < len(reactantImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    parts.append(makeSeparatorImage("→", maxH, width=100))
    for i, img in enumerate(productImgs):
        parts.append(img)
        if i < len(productImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    totalW = sum(p.width for p in parts) + gap * (len(parts) - 1)
    canvas = Image.new("RGB", (totalW, maxH), "white")
    x = 0
    for part in parts:
        canvas.paste(part, (x, (maxH - part.height) // 2))
        x += part.width + gap
    titleBanner = makeTextBanner(panelTitle, canvas.width, bannerHeight=38)
    final = Image.new("RGB", (canvas.width, canvas.height + titleBanner.height + 10), "white")
    final.paste(titleBanner, (0, 0))
    final.paste(canvas,      (0, titleBanner.height + 10))
    return final

# Feasible-reaction network (all steps here are DORA-XGB high-feasibility by construction)
feasibleRxnDF = reactionDF_highFeasible.copy()
if "reactionString" not in feasibleRxnDF.columns:
    feasibleRxnDF["reactionString"] = (
        feasibleRxnDF["reactants"].astype(str) + " >> " + feasibleRxnDF["products"].astype(str)
    )

# canonical reactant -> feasible reactions consuming it
reactantToFeasibleRxns = defaultdict(list)
for idx, row in feasibleRxnDF.iterrows():
    for smi in splitMoleculeString(row["reactants"]):
        reactantToFeasibleRxns[canonicalizeSmiles(smi)].append(idx)

def carrierProductsOf(row):
    """Non-cofactor, non-helper, non-starter products that can be transformed further."""
    out = []
    for productSmi in splitMoleculeString(row["products"]):
        productCanon = canonicalizeSmiles(productSmi)
        if productCanon in canonHelperSet:
            continue
        if productCanon in canonStarterSet:
            continue
        productMol = Chem.MolFromSmiles(str(productSmi))
        if isCofactorOrEndogenous(productMol, productCanon, cofactorExclusionSet, physicochemicalCutoffs):
            continue
        out.append(productCanon)
    return out

# Depth-capped BFS over feasible edges: shortest fully-feasible route (<= doranet_generations) to each compound.
# parent[compound] = (rxnIdx, precursorCanon); absence means "no all-feasible route within doranet_generations".
parentReaction = {}
seenCompounds  = set(canonStarterSet)
searchQueue    = deque((c, 0) for c in canonStarterSet)

while searchQueue:
    currentCanon, currentDepth = searchQueue.popleft()
    if currentDepth >= doranet_generations:
        continue
    for rxnIdx in reactantToFeasibleRxns.get(currentCanon, []):
        row = feasibleRxnDF.loc[rxnIdx]
        for productCanon in carrierProductsOf(row):
            if productCanon in seenCompounds:
                continue
            seenCompounds.add(productCanon)
            parentReaction[productCanon] = (rxnIdx, currentCanon)
            searchQueue.append((productCanon, currentDepth + 1))

def feasibleRouteTo(targetCanon):
    """Reaction strings of the fully-feasible route to a target, forward order; [] if none within doranet_generations."""
    steps = []
    cur   = targetCanon
    while cur in parentReaction:
        rxnIdx, precursorCanon = parentReaction[cur]
        steps.append(feasibleRxnDF.loc[rxnIdx, "reactionString"])
        cur = precursorCanon
    steps.reverse()
    return steps

def reorderStepsForward(steps, canonStarterSet):
    if len(steps) <= 1:
        return steps
    parsed = []
    for smi in steps:
        if ">>" not in str(smi):
            continue
        rStr, pStr = str(smi).split(">>", 1)
        parsed.append({
            "smi"      : smi,
            "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
            "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
        })
    if not parsed:
        return steps
    firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
    if firstIdx is None:
        return steps
    ordered   = [parsed[firstIdx]]
    remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["products"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return [s["smi"] for s in ordered]

def buildRouteImageFromSteps(stepReactions, starterCanon, targetCanon,
                             headerText, subImageSize, bannerWidth):
    canonStarterSetLocal = {starterCanon}
    numSteps             = len(stepReactions)
    forwardSteps         = reorderStepsForward(stepReactions, canonStarterSetLocal)
    panelBlocks = [makeTextBanner(headerText, bannerWidth=bannerWidth, bannerHeight=44, bgColor="#eef2f7")]
    for stepIdx, rxnSmi in enumerate(forwardSteps, start=1):
        panel = makeCustomReactionPanel(
            rxnSmi, canonStarterSetLocal, targetCanon,
            panelTitle=f"Reaction step {stepIdx} of {numSteps}",
            subImageSize=subImageSize,
        )
        if panel:
            panelBlocks.append(panel)
    return stackPanelsVertically(panelBlocks, gapSize=14, bgColor="white")

if saveFigures:
    os.makedirs(figureOutDir, exist_ok=True)

candidateDF = generatedCompoundsDF_highFeasibility.reset_index(drop=True)

# Keep only candidates that have a fully-feasible route within doranet_generations
feasibleRoutes = []
for _, cand in candidateDF.iterrows():
    targetCanon = canonicalizeSmiles(cand["Canonical_SMILES"])
    steps       = feasibleRouteTo(targetCanon)
    if steps and len(steps) <= doranet_generations:
        firstReactants = str(steps[0]).split(">>", 1)[0]
        starterCanon   = next(
            (canonicalizeSmiles(s) for s in firstReactants.split(".") if canonicalizeSmiles(s) in canonStarterSet),
            cand.get("starter_Canonical_SMILES", ""),
        )
        feasibleRoutes.append((len(steps), targetCanon, starterCanon, steps))

feasibleRoutes.sort(key=lambda t: t[0])

print(f"High-feasibility candidates                 : {len(candidateDF):,}")
print(f"With a fully-feasible route (<= {doranet_generations} steps) : {len(feasibleRoutes):,}")
stepHistogram = defaultdict(int)
for nSteps, *_ in feasibleRoutes:
    stepHistogram[nSteps] += 1
for nSteps in sorted(stepHistogram):
    print(f"  {nSteps}-step feasible pathways : {stepHistogram[nSteps]:,}")

drawnByStep = defaultdict(int)
for numSteps, targetCanon, starterCanon, steps in feasibleRoutes:
    if drawnByStep[numSteps] >= numToDisplayPerStep:
        continue
    drawnByStep[numSteps] += 1
    headerText = f"Starter: {str(starterCanon)[:40]}  |  Target: {targetCanon[:50]}  |  Steps: {numSteps}"
    routeImage = buildRouteImageFromSteps(steps, starterCanon, targetCanon, headerText, subImgSize, bannerWidth=1800)
    if routeImage is None:
        continue
    print(f"  [{numSteps}-step #{drawnByStep[numSteps]}] {targetCanon}")
    display(routeImage)
    if saveFigures:
        basePath         = os.path.join(figureOutDir, f"highFeasPathway_{numSteps}step_{drawnByStep[numSteps]}")
        pngPath, pdfPath = saveImageHighRes(routeImage, basePath, outputDpi)
        print(f"      saved: {pngPath}")

print("\nDone.")

### Feasible pathways (horizontal cascade with feasibility scores)

In [ ]:
# CONFIG
numToDisplay           = 5
selectedRoutes         = {}   # empty: every step-count group shows its first numToDisplay routes
layoutDirection        = "horizontal"
moleculeImageSizePx    = (900, 680)
arrowSpanPx            = 520
outputDpi              = 1200
showStarterTargetTags  = True
showCofactorLabels     = True
showFeasibilityScore   = True
feasibilityScoreColumn = bestRuleScoreCol   # column in reactionDF_DORAXGB_highFeasibility
feasibilityScorePrefix = ""                          # set to "feasibility = " to prefix the number
saveFigures            = True
maxCofactorHeavyAtoms  = 12   # unknown species larger than this are treated as co-products, not cofactors

# Optional display-name overrides, keyed by the Name in all_cofactors.tsv.
# Leave empty to show the table names verbatim (CO-A, OXYGEN-MOLECULE, ...).
cofactorDisplayOverride = {
    "CO-A": "CoA",
    "ACETYL-COA": "Acetyl-CoA",
    "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O",
    "HYDROGEN-PEROXIDE": "H2O2",
    "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3",
    "S-ADENOSYLMETHIONINE": "SAM",
    "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate",
    "GLT": "Glutamate",
    "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

captionFont     = loadFont(30)
cofactorFont    = loadFont(26)
feasibilityFont = loadFont(28)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    """
    Canonical, stereo- and charge-insensitive key. Tolerates '*' wildcard atoms
    in the cofactor table (FAD, F420, quinones, formyl-THF).
    """
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    """Map canonical SMILES key -> display name, sourced from all_cofactors.tsv."""
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactors will fall back to formulas.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def isKnownCofactor(smi):
    return cofactorKey(smi) in cofactorNameByKey

def labelSpecies(smi):
    key = cofactorKey(smi)
    if key in cofactorNameByKey:
        return cofactorNameByKey[key]
    mol = Chem.MolFromSmiles(str(smi))
    return rdMolDescriptors.CalcMolFormula(mol) if mol else str(smi)

# Rule-level role tokens, used only when a reaction carries no explicit cofactor molecule.
cofactorTokenNameMap = {
    "WATER": "H2O", "HYDROGEN_PEROXIDE": "H2O2", "OXYGEN": "O2", "O2": "O2",
    "CARBON_DIOXIDE": "CO2", "AMMONIA": "NH3", "AMMONIUM": "NH4+",
    "NAD": "NAD+", "NAD_PLUS": "NAD+", "NADH": "NADH",
    "NADP": "NADP+", "NADP_PLUS": "NADP+", "NADPH": "NADPH",
    "L_GLUTAMATE": "Glutamate", "GLUTAMATE": "Glutamate",
    "2_KETOGLUTARATE": "2-Ketoglutarate", "2_OXOGLUTARATE": "2-Oxoglutarate",
    "METHYL_DONOR_COF": "SAM", "METHYL_ACCEPTOR_COF": "SAH",
    "ATP": "ATP", "ADP": "ADP", "AMP": "AMP",
    "PHOSPHATE": "Pi", "DIPHOSPHATE": "PPi", "COA": "CoA", "ACETYL_COA": "Acetyl-CoA",
}
primaryRoleTokens = {"ANY", "ANY_COF", ""}

def heavyAtomCount(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return mol.GetNumHeavyAtoms() if mol else 0

def splitStarters(starterStr):
    return [] if pd.isna(starterStr) else [x.strip() for x in str(starterStr).split(";") if x.strip()]

def dedupeByCanon(smilesList):
    seen, kept = set(), []
    for s in smilesList:
        if not s:
            continue
        c = canonicalizeSmiles(s)
        if c not in seen:
            seen.add(c)
            kept.append(s)
    return kept

def dedupePreserveOrder(items):
    seen, kept = set(), []
    for x in items:
        if x and x not in seen:
            seen.add(x)
            kept.append(x)
    return kept

def cleanRoleToken(token):
    key = token.strip().upper().replace("-", "_").replace(" ", "_")
    if key in cofactorTokenNameMap:
        return cofactorTokenNameMap[key]
    return token.strip().replace("_", " ").title()

def extractRuleCofactors(cellValue):
    tokens = [t.strip() for t in re.split(r"[;,]", str(cellValue)) if t.strip()]
    labels = [cleanRoleToken(t) for t in tokens if t.strip().upper() not in primaryRoleTokens]
    return dedupePreserveOrder(labels)

def buildRuleCofactorMap(path):
    ruleMap = {}
    if not os.path.exists(path):
        print(f"  ruleset not found at {path}, using reaction-level cofactors only.")
        return ruleMap
    ruleset = pd.read_csv(path, sep="\t")
    nameCol = "Name" if "Name" in ruleset.columns else ruleset.columns[0]
    hasReact, hasProd = "Reactants" in ruleset.columns, "Products" in ruleset.columns
    for _, row in ruleset.iterrows():
        name = str(row.get(nameCol, "")).strip()
        if not name:
            continue
        entry = {
            "consumed": extractRuleCofactors(row["Reactants"]) if hasReact else [],
            "produced": extractRuleCofactors(row["Products"]) if hasProd else [],
        }
        ruleMap.setdefault(name, entry)
        ruleMap.setdefault(name.split("_")[0], entry)
    return ruleMap

def ruleCofactorsForStep(ruleName, ruleCofactorMap):
    name = str(ruleName).strip()
    entry = ruleCofactorMap.get(name) or ruleCofactorMap.get(name.split("_")[0])
    return (entry["consumed"], entry["produced"]) if entry else ([], [])

def parseReactionSides(reactionSmiles):
    if ">>" not in str(reactionSmiles):
        return [], []
    rStr, pStr = str(reactionSmiles).split(">>", 1)
    return (
        [s.strip() for s in rStr.split(".") if s.strip()],
        [s.strip() for s in pStr.split(".") if s.strip()],
    )

def reactionKey(reactionSmiles):
    """Order- and spacing-independent reaction identity: canonical reactants and products."""
    reactants, products = parseReactionSides(reactionSmiles)
    reactantKey = ".".join(sorted(canonicalizeSmiles(s) for s in reactants))
    productKey  = ".".join(sorted(canonicalizeSmiles(s) for s in products))
    return reactantKey + ">>" + productKey

def buildFeasibilityScoreMap(scoreDF):
    """Map canonical reactionKey -> feasibilityScore_rule3 from reactionDF_DORAXGB_highFeasibility."""
    if feasibilityScoreColumn not in scoreDF.columns:
        raise KeyError(f"Column '{feasibilityScoreColumn}' not in reactionDF_DORAXGB_highFeasibility; available: {list(scoreDF.columns)}")
    if "reactionString" in scoreDF.columns:
        reactionSeries = scoreDF["reactionString"].astype(str)
    elif "reactants" in scoreDF.columns and "products" in scoreDF.columns:
        reactionSeries = scoreDF["reactants"].astype(str) + " >> " + scoreDF["products"].astype(str)
    else:
        raise KeyError("reactionDF_DORAXGB_highFeasibility needs a 'reactionString' column or both 'reactants' and 'products' columns to key reactions.")
    scoreByReaction = {}
    for reactionStr, score in zip(reactionSeries, scoreDF[feasibilityScoreColumn]):
        try:
            scoreValue = float(score)
        except (TypeError, ValueError):
            continue
        scoreByReaction.setdefault(reactionKey(reactionStr), scoreValue)
    return scoreByReaction

def reorderStepDictsForward(stepDicts, canonStarterSet):
    if len(stepDicts) <= 1:
        return stepDicts
    enriched = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants and not products:
            continue
        enriched.append({
            **sd,
            "reactantSet": {canonicalizeSmiles(s) for s in reactants},
            "productSet" : {canonicalizeSmiles(s) for s in products},
        })
    if not enriched:
        return stepDicts
    firstIdx = next((i for i, s in enumerate(enriched) if s["reactantSet"] & canonStarterSet), None)
    if firstIdx is None:
        return stepDicts
    ordered   = [enriched[firstIdx]]
    remaining = [s for i, s in enumerate(enriched) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["productSet"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactantSet"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return ordered

def pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants):
    # never let a known cofactor become a backbone node
    pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon and not isKnownCofactor(p)]
    if not pool:
        pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon] or products
    carrier = next((p for p in pool if canonicalizeSmiles(p) == canonTargetSmi), None)
    if carrier is None and nextReactants is not None:
        nextCanon = {canonicalizeSmiles(s) for s in nextReactants}
        feeding = [p for p in pool if canonicalizeSmiles(p) in nextCanon]
        if feeding:
            carrier = max(feeding, key=heavyAtomCount)
    if carrier is None:
        carrier = max(pool, key=heavyAtomCount)
    return carrier

def resolveCofactorLabels(step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap):
    def keepAsCofactor(smi):
        return isKnownCofactor(smi) or heavyAtomCount(smi) <= maxCofactorHeavyAtoms

    consumedSpecies = [s for s in dedupeByCanon([r for r in reactants if canonicalizeSmiles(r) != prevCarrierCanon]) if keepAsCofactor(s)]
    producedSpecies = [s for s in dedupeByCanon([p for p in products  if canonicalizeSmiles(p) != carrierCanon])   if keepAsCofactor(s)]
    consumedLabels = [labelSpecies(s) for s in consumedSpecies]
    producedLabels = [labelSpecies(s) for s in producedSpecies]

    ruleConsumed, ruleProduced = ruleCofactorsForStep(step.get("ruleName", ""), ruleCofactorMap)
    if not consumedLabels and not producedLabels and (ruleConsumed or ruleProduced):
        consumedLabels, producedLabels = ruleConsumed, ruleProduced

    return dedupePreserveOrder(consumedLabels), dedupePreserveOrder(producedLabels)

def buildBackbone(stepDicts, canonStarterSet, canonTargetSmi, ruleCofactorMap):
    steps = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants or not products:
            continue
        steps.append({**sd, "reactants": reactants, "products": products})
    if not steps:
        return [], []

    firstReactants = steps[0]["reactants"]
    startCarrier = next((s for s in firstReactants if canonicalizeSmiles(s) in canonStarterSet), None)
    if startCarrier is None:
        nonCofactorFirst = [s for s in firstReactants if not isKnownCofactor(s)] or firstReactants
        startCarrier = max(nonCofactorFirst, key=heavyAtomCount)
    nodes            = [startCarrier]
    edges            = []
    prevCarrierCanon = canonicalizeSmiles(startCarrier)

    for stepIdx, step in enumerate(steps):
        reactants, products = step["reactants"], step["products"]
        nextReactants = steps[stepIdx + 1]["reactants"] if stepIdx + 1 < len(steps) else None
        carrier      = pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants)
        carrierCanon = canonicalizeSmiles(carrier)
        consumedLabels, producedLabels = resolveCofactorLabels(
            step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap
        )
        nodes.append(carrier)
        edges.append({
            "consumed"         : consumedLabels,
            "produced"         : producedLabels,
            "feasibilityScore" : step.get("feasibilityScore"),
        })
        prevCarrierCanon = carrierCanon
    return nodes, edges

def drawMoleculeCairo(smi, sizePx):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    AllChem.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DCairo(sizePx[0], sizePx[1])
    opts = drawer.drawOptions()
    opts.bondLineWidth   = 3
    opts.padding         = 0.10
    opts.clearBackground = True
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    return Image.open(BytesIO(drawer.GetDrawingText())).convert("RGB")

def textWidth(draw, text, font):
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        return bbox[2] - bbox[0]
    except Exception:
        return len(text) * 12

def centeredTextX(draw, text, boxWidth, y, font, color="#333333"):
    draw.text(((boxWidth - textWidth(draw, text, font)) // 2, y), text, fill=color, font=font)

def makeCaptionBanner(text, width, height=46):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    centeredTextX(draw, text, width, (height - 34) // 2, captionFont)
    return img

def makeMoleculePanel(smi, caption, imageSizePx):
    molImg = drawMoleculeCairo(smi, imageSizePx)
    if molImg is None:
        return None
    if not caption:
        return molImg
    banner = makeCaptionBanner(caption, molImg.width)
    panel  = Image.new("RGB", (molImg.width, molImg.height + banner.height), "white")
    panel.paste(molImg, (0, 0))
    panel.paste(banner, (0, molImg.height))
    return panel

def joinLabels(labelList):
    return " + ".join([l for l in labelList if l])

def putLabelCentered(draw, text, centerX, y, font, panelWidth, color="#555555"):
    tw = textWidth(draw, text, font)
    x  = int(centerX - tw / 2)
    x  = max(4, min(x, panelWidth - tw - 4))
    draw.text((x, y), text, fill=color, font=font)

def makeHorizontalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cy    = height // 2
    left  = int(width * 0.10)
    right = int(width * 0.90)

    # main straight reaction arrow (left to right)
    draw.line([(left, cy), (right, cy)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.045)
    draw.polygon(
        [(right - int(head * 2.2), cy - head), (right - int(head * 2.2), cy + head), (right, cy)],
        fill=color,
    )

    # feasibility score, centered just below the main arrow, independent of cofactor labels
    if showFeasibilityScore and feasibilityText:
        scoreY = cy + int(0.12 * height)
        putLabelCentered(draw, feasibilityText, (left + right) // 2, scoreY, feasibilityFont, width, color="#1a1a1a")

    if not showCofactorLabels or (not consumedText and not producedText):
        return img

    # curved feeder: reactant enters at the left (tail) end, curve dips down to the
    # main arrow, product leaves at the right (arrowhead) end. Drawn as a quadratic
    # bezier so the endpoints sit under the two labels and the head follows the tangent.
    span  = right - left
    xL    = int(left + 0.15 * span)     # tail end, under the consumed label
    xR    = int(right - 0.05 * span)    # head end, under the produced label
    yTop  = cy - int(0.30 * height)     # both ends sit above the main arrow
    p0    = (xL, yTop)
    ctrl  = ((xL + xR) / 2.0, cy + 0.34 * height)   # control below pulls the curve down to the arrow
    p1    = (xR, yTop)

    thin = max(3, lineWidth // 2)
    pts  = []
    steps = 48
    for i in range(steps + 1):
        t  = i / steps
        mt = 1.0 - t
        x  = mt * mt * p0[0] + 2 * mt * t * ctrl[0] + t * t * p1[0]
        y  = mt * mt * p0[1] + 2 * mt * t * ctrl[1] + t * t * p1[1]
        pts.append((x, y))
    draw.line(pts, fill=color, width=thin, joint="curve")

    # arrowhead at the product (right) end, oriented along the curve tangent ctrl -> p1
    dx, dy = p1[0] - ctrl[0], p1[1] - ctrl[1]
    norm   = math.hypot(dx, dy) or 1.0
    ux, uy = dx / norm, dy / norm
    px, py = -uy, ux
    ah     = int(height * 0.05)
    tip    = p1
    base   = (p1[0] - ux * ah * 1.7, p1[1] - uy * ah * 1.7)
    draw.polygon([
        tip,
        (base[0] + px * ah, base[1] + py * ah),
        (base[0] - px * ah, base[1] - py * ah),
    ], fill=color)

    # labels above each end: consumed at the tail (left), produced at the head (right)
    labelY = yTop - 40
    if consumedText:
        putLabelCentered(draw, consumedText, xL, labelY, cofactorFont, width)
    if producedText:
        putLabelCentered(draw, producedText, xR, labelY, cofactorFont, width)
    return img

def makeVerticalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cx     = width // 2
    top    = int(height * 0.14)
    bottom = int(height * 0.86)
    draw.line([(cx, top), (cx, bottom)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.10)
    draw.polygon(
        [(cx - head, bottom - int(head * 1.6)), (cx + head, bottom - int(head * 1.6)), (cx, bottom)],
        fill=color,
    )
    if showFeasibilityScore and feasibilityText:
        putLabelCentered(draw, feasibilityText, cx, bottom + 6, feasibilityFont, width, color="#1a1a1a")
    midY = (top + bottom) // 2
    if showCofactorLabels and consumedText:
        draw.text((cx + 18, midY - 40), consumedText, fill="#555555", font=cofactorFont)
    if showCofactorLabels and producedText:
        draw.text((cx + 18, midY + 8), producedText, fill="#555555", font=cofactorFont)
    return img

def stackPanelsHorizontally(parts, bgColor="white"):
    valid = [p for p in parts if p is not None]
    if not valid:
        return None
    w = sum(p.width for p in valid)
    h = max(p.height for p in valid)
    canvas = Image.new("RGB", (w, h), bgColor)
    x = 0
    for p in valid:
        canvas.paste(p, (x, 0))
        x += p.width
    return canvas

def stackPanelsVertically(parts, bgColor="white"):
    valid = [p for p in parts if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height
    return canvas

def buildCascadeImage(rowData, imageSizePx, arrowSpanPx, direction, ruleCofactorMap):
    canonStarterSet = {canonicalizeSmiles(s) for s in splitStarters(rowData.get("starter_Canonical_SMILES", ""))}
    canonTargetSmi  = canonicalizeSmiles(rowData.get("targetSMILES", ""))
    orderedSteps    = reorderStepDictsForward(rowData["stepDicts"], canonStarterSet)
    nodes, edges    = buildBackbone(orderedSteps, canonStarterSet, canonTargetSmi, ruleCofactorMap)
    if len(nodes) < 2:
        return None

    parts   = []
    lastIdx = len(nodes) - 1
    for idx, smi in enumerate(nodes):
        caption = ""
        if showStarterTargetTags and idx == 0:
            caption = "Starter"
        elif showStarterTargetTags and idx == lastIdx:
            caption = "Target"
        parts.append(makeMoleculePanel(smi, caption, imageSizePx))
        if idx != lastIdx:
            edge            = edges[idx]
            consumedText    = joinLabels(edge["consumed"])
            producedText    = joinLabels(edge["produced"])
            score           = edge.get("feasibilityScore")
            feasibilityText = f"{feasibilityScorePrefix}{score:.3f}" if score is not None else ""
            if direction == "horizontal":
                parts.append(makeHorizontalArrow(arrowSpanPx, imageSizePx[1], consumedText, producedText, feasibilityText))
            else:
                parts.append(makeVerticalArrow(imageSizePx[0], arrowSpanPx, consumedText, producedText, feasibilityText))

    return stackPanelsHorizontally(parts) if direction == "horizontal" else stackPanelsVertically(parts)

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

ruleCofactorMap = buildRuleCofactorMap(rulesetPath)
scoreByReaction = buildFeasibilityScoreMap(reactionDF_DORAXGB_highFeasibility)

if saveFigures:
    os.makedirs(articleFiguresDir, exist_ok=True)

# Build route-level records directly from the BFS output. feasibleRoutes is a list of
# (numSteps, targetCanon, starterCanon, steps) where steps is a forward-ordered list of
# "reactants >> products" strings. Each step's feasibilityScore_rule3 is looked up by a
# canonical reactionKey so ordering and spacing differences do not break the match.
routeRecords = []
for routeIndex, (numSteps, targetCanon, starterCanon, steps) in enumerate(feasibleRoutes):
    stepDicts = [
        {
            "reactionSmiles"   : str(rxnString),
            "ruleName"         : "",
            "feasibilityScore" : scoreByReaction.get(reactionKey(str(rxnString))),
        }
        for rxnString in steps
    ]
    routeRecords.append({
        "routeId"                  : f"bfsRoute_{routeIndex:04d}",
        "numSteps"                 : int(numSteps),
        "starter_Canonical_SMILES" : starterCanon,
        "targetSMILES"             : targetCanon,
        "stepDicts"                : stepDicts,
    })

routeLevelDF = (
    pd.DataFrame(
        routeRecords,
        columns=["routeId", "numSteps", "starter_Canonical_SMILES", "targetSMILES", "stepDicts"],
    )
    .sort_values(["numSteps", "routeId"])
    .reset_index(drop=True)
)

totalSteps        = sum(len(rec["stepDicts"]) for rec in routeRecords)
missingScoreSteps = sum(1 for rec in routeRecords for sd in rec["stepDicts"] if sd.get("feasibilityScore") is None)
if missingScoreSteps:
    print(f"Steps with no matched {feasibilityScoreColumn}: {missingScoreSteps:,} of {totalSteps:,}")

for numSteps, groupDF in routeLevelDF.groupby("numSteps"):
    availableRoutes = groupDF.reset_index(drop=True)
    indices         = selectedRoutes.get(numSteps, list(range(numToDisplay)))
    validIndices    = [i for i in indices if i < len(availableRoutes)]

    print(f"\n{numSteps}-step pathways: {len(availableRoutes):,} available, displaying {len(validIndices):,}")
    if not validIndices:
        print(f"  No valid indices for {numSteps}-step, check selectedRoutes config.")
        continue

    for rank, idx in enumerate(validIndices, start=1):
        rowData      = availableRoutes.loc[idx]
        cascadeImage = buildCascadeImage(rowData, moleculeImageSizePx, arrowSpanPx, layoutDirection, ruleCofactorMap)
        if cascadeImage is None:
            print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']} skipped, no drawable chain.")
            continue
        print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']}")
        display(cascadeImage)
        if saveFigures:
            basePath         = os.path.join(articleFiguresDir, f"{fileNamePrefix}_pathway_{rowData['routeId']}_{numSteps}step")
            pngPath, pdfPath = saveImageHighRes(cascadeImage, basePath, outputDpi)
            print(f"      saved: {pngPath}")
            print(f"      saved: {pdfPath}")

print("\nDone.")

## Plot pathways for all starter compounds

These renderers reproduce the original notebook and now run for **every**
starter. The helper cells below define the drawing routines once; the final
cell loops over all starters, and for each one shows the feasible pathways
from that starter to its high-feasibility target compounds, using that
starter's own network depth from its `doranet_config.yaml`.

### Pathway rendering helpers (vertical, per reaction step)

In [18]:
# CONFIG
numToDisplayPerStep = 4
saveVerticalFigures         = False
figureOutDir        = os.path.join(resultsDir, "PathwayFigures")
subImgSize          = (600, 300)
articleFiguresDir     = os.path.join(resultsDir, "PathwayFigures")
rulesetPath           = rulesetPathForPlots
cofactorTablePath     = cofactorTablePathForPlots
gapSize = 14

cofactorDisplayOverride = {
    "CO-A": "CoA", "ACETYL-COA": "Acetyl-CoA", "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O", "HYDROGEN-PEROXIDE": "H2O2", "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3", "S-ADENOSYLMETHIONINE": "SAM", "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate", "GLT": "Glutamate", "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

separatorFont = loadFont(52)
labelFont     = loadFont(18)
headerFont    = loadFont(22)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactor names will not be applied.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def cofactorNameFor(smi):
    return cofactorNameByKey.get(cofactorKey(smi))

def rdkitImageToPil(imageObj):
    if isinstance(imageObj, Image.Image):
        return imageObj.convert("RGB")
    if isinstance(imageObj, (bytes, bytearray)):
        return Image.open(BytesIO(imageObj)).convert("RGB")
    return Image.open(BytesIO(imageObj.data)).convert("RGB")

def centeredText(draw, text, imgWidth, y, font, color="black"):
    try:
        bbox  = draw.textbbox((0, 0), text, font=font)
        textW = bbox[2] - bbox[0]
    except Exception:
        textW = len(text) * 8
    draw.text(((imgWidth - textW) // 2, y), text, fill=color, font=font)

def makeTextBanner(text, bannerWidth, bannerHeight=40, bgColor="white", font=None):
    img = Image.new("RGB", (bannerWidth, bannerHeight), bgColor)
    centeredText(ImageDraw.Draw(img), text, bannerWidth, (bannerHeight - 24) // 2, font or headerFont)
    return img

def stackPanelsVertically(panelList, gapSize=14, bgColor="white"):
    valid = [p for p in panelList if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid) + gapSize * (len(valid) - 1)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height + gapSize
    return canvas

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

def reorderStepsForward(steps, canonStarterSet):
    if len(steps) <= 1:
        return steps
    parsed = []
    for smi in steps:
        if ">>" not in str(smi):
            continue
        rStr, pStr = str(smi).split(">>", 1)
        parsed.append({
            "smi"      : smi,
            "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
            "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
        })
    if not parsed:
        return steps
    firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
    if firstIdx is None:
        return steps
    ordered   = [parsed[firstIdx]]
    remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["products"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return [s["smi"] for s in ordered]

def getMolLabel(smi, canonStarterSet, canonTargetSmi, side="reactant"):
    canon = canonicalizeSmiles(smi)
    if canon in canonStarterSet:
        return "Starter compound"
    if canon == canonTargetSmi:
        return "Target compound"
    cofactorName = cofactorNameFor(smi)
    if cofactorName:
        return cofactorName
    return "Reactant" if side == "reactant" else "Product"

def makeLabeledMolImage(smi, label, subImageSize=(280, 210)):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    molImg      = rdkitImageToPil(Draw.MolToImage(mol, size=subImageSize))
    labelBanner = Image.new("RGB", (molImg.width, 34), "#f0f0f0")
    centeredText(ImageDraw.Draw(labelBanner), label, molImg.width, 7, labelFont, "#333333")
    combined = Image.new("RGB", (molImg.width, molImg.height + labelBanner.height), "white")
    combined.paste(molImg,      (0, 0))
    combined.paste(labelBanner, (0, molImg.height))
    return combined

def makeSeparatorImage(text, height, width=90):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    try:
        bbox         = draw.textbbox((0, 0), text, font=separatorFont)
        textW, textH = bbox[2] - bbox[0], bbox[3] - bbox[1]
    except Exception:
        textW, textH = 30, 50
    draw.text(((width - textW) // 2, (height - textH) // 2), text, fill="#222222", font=separatorFont)
    return img

def makeCustomReactionPanel(rxnSmi, canonStarterSet, canonTargetSmi,
                             panelTitle, subImageSize=(280, 210)):
    if ">>" not in str(rxnSmi):
        return None
    reactantStr, productStr = str(rxnSmi).split(">>", 1)
    reactantSmiles = (
        [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) in canonStarterSet]
        + [s for s in [x.strip() for x in reactantStr.split(".") if x.strip()] if canonicalizeSmiles(s) not in canonStarterSet]
    )
    productSmiles = (
        [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) != canonTargetSmi]
        + [s for s in [x.strip() for x in productStr.split(".") if x.strip()] if canonicalizeSmiles(s) == canonTargetSmi]
    )
    reactantImgs = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="reactant"), subImageSize)
        for s in reactantSmiles] if img is not None]
    productImgs  = [img for img in [
        makeLabeledMolImage(s, getMolLabel(s, canonStarterSet, canonTargetSmi, side="product"), subImageSize)
        for s in productSmiles]  if img is not None]
    if not reactantImgs or not productImgs:
        return None
    maxH  = max(img.height for img in reactantImgs + productImgs)
    gap   = 16
    parts = []
    for i, img in enumerate(reactantImgs):
        parts.append(img)
        if i < len(reactantImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    parts.append(makeSeparatorImage("→", maxH, width=100))
    for i, img in enumerate(productImgs):
        parts.append(img)
        if i < len(productImgs) - 1:
            parts.append(makeSeparatorImage("+", maxH, width=80))
    totalW = sum(p.width for p in parts) + gap * (len(parts) - 1)
    canvas = Image.new("RGB", (totalW, maxH), "white")
    x = 0
    for part in parts:
        canvas.paste(part, (x, (maxH - part.height) // 2))
        x += part.width + gap
    titleBanner = makeTextBanner(panelTitle, canvas.width, bannerHeight=38)
    final = Image.new("RGB", (canvas.width, canvas.height + titleBanner.height + 10), "white")
    final.paste(titleBanner, (0, 0))
    final.paste(canvas,      (0, titleBanner.height + 10))
    return final


### Pathway rendering helpers (horizontal cascade)

In [15]:
# CONFIG
numToDisplay           = 5
selectedRoutes         = {}   # empty: every step-count group shows its first numToDisplay routes
layoutDirection        = "horizontal"
moleculeImageSizePx    = (900, 680)
arrowSpanPx            = 520
outputDpi              = 1200
showStarterTargetTags  = True
showCofactorLabels     = True
showFeasibilityScore   = True
feasibilityScoreColumn = feasibilityScoreCol   # column in reactionDF_DORAXGB_highFeasibility
feasibilityScorePrefix = ""                          # set to "feasibility = " to prefix the number
saveCascadeFigures            = True
maxCofactorHeavyAtoms  = 12   # unknown species larger than this are treated as co-products, not cofactors

# Optional display-name overrides, keyed by the Name in all_cofactors.tsv.
# Leave empty to show the table names verbatim (CO-A, OXYGEN-MOLECULE, ...).
cofactorDisplayOverride = {
    "CO-A": "CoA",
    "ACETYL-COA": "Acetyl-CoA",
    "OXYGEN-MOLECULE": "O2",
    "WATER": "H2O",
    "HYDROGEN-PEROXIDE": "H2O2",
    "CARBON-DIOXIDE": "CO2",
    "AMMONIA": "NH3",
    "S-ADENOSYLMETHIONINE": "SAM",
    "ADENOSYL-HOMO-CYS": "SAH",
    "2-KETOGLUTARATE": "2-Ketoglutarate",
    "GLT": "Glutamate",
    "PROTON": "H+",
}

def loadFont(size):
    for path in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
    ]:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

captionFont     = loadFont(30)
cofactorFont    = loadFont(26)
feasibilityFont = loadFont(28)

def canonicalizeSmiles(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol else str(smi)

def cofactorKey(smi):
    """
    Canonical, stereo- and charge-insensitive key. Tolerates '*' wildcard atoms
    in the cofactor table (FAD, F420, quinones, formyl-THF).
    """
    mol = Chem.MolFromSmiles(str(smi), sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except Exception:
        return None

def loadCofactorNames(path):
    """Map canonical SMILES key -> display name, sourced from all_cofactors.tsv."""
    nameByKey = {}
    if not os.path.exists(path):
        print(f"Cofactor table not found at {path}; cofactors will fall back to formulas.")
        return nameByKey
    table = pd.read_csv(path, sep="\t")
    if "SMILES" not in table.columns or "Name" not in table.columns:
        print(f"Cofactor table {path} lacks Name/SMILES columns; names not applied.")
        return nameByKey
    for _, row in table.iterrows():
        key = cofactorKey(row["SMILES"])
        if key is None:
            continue
        rawName = str(row["Name"]).strip()
        nameByKey.setdefault(key, cofactorDisplayOverride.get(rawName, rawName))
    return nameByKey

cofactorNameByKey = loadCofactorNames(cofactorTablePath)

def isKnownCofactor(smi):
    return cofactorKey(smi) in cofactorNameByKey

def labelSpecies(smi):
    key = cofactorKey(smi)
    if key in cofactorNameByKey:
        return cofactorNameByKey[key]
    mol = Chem.MolFromSmiles(str(smi))
    return rdMolDescriptors.CalcMolFormula(mol) if mol else str(smi)

# Rule-level role tokens, used only when a reaction carries no explicit cofactor molecule.
cofactorTokenNameMap = {
    "WATER": "H2O", "HYDROGEN_PEROXIDE": "H2O2", "OXYGEN": "O2", "O2": "O2",
    "CARBON_DIOXIDE": "CO2", "AMMONIA": "NH3", "AMMONIUM": "NH4+",
    "NAD": "NAD+", "NAD_PLUS": "NAD+", "NADH": "NADH",
    "NADP": "NADP+", "NADP_PLUS": "NADP+", "NADPH": "NADPH",
    "L_GLUTAMATE": "Glutamate", "GLUTAMATE": "Glutamate",
    "2_KETOGLUTARATE": "2-Ketoglutarate", "2_OXOGLUTARATE": "2-Oxoglutarate",
    "METHYL_DONOR_COF": "SAM", "METHYL_ACCEPTOR_COF": "SAH",
    "ATP": "ATP", "ADP": "ADP", "AMP": "AMP",
    "PHOSPHATE": "Pi", "DIPHOSPHATE": "PPi", "COA": "CoA", "ACETYL_COA": "Acetyl-CoA",
}
primaryRoleTokens = {"ANY", "ANY_COF", ""}

def heavyAtomCount(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return mol.GetNumHeavyAtoms() if mol else 0

def splitStarters(starterStr):
    return [] if pd.isna(starterStr) else [x.strip() for x in str(starterStr).split(";") if x.strip()]

def dedupeByCanon(smilesList):
    seen, kept = set(), []
    for s in smilesList:
        if not s:
            continue
        c = canonicalizeSmiles(s)
        if c not in seen:
            seen.add(c)
            kept.append(s)
    return kept

def dedupePreserveOrder(items):
    seen, kept = set(), []
    for x in items:
        if x and x not in seen:
            seen.add(x)
            kept.append(x)
    return kept

def cleanRoleToken(token):
    key = token.strip().upper().replace("-", "_").replace(" ", "_")
    if key in cofactorTokenNameMap:
        return cofactorTokenNameMap[key]
    return token.strip().replace("_", " ").title()

def extractRuleCofactors(cellValue):
    tokens = [t.strip() for t in re.split(r"[;,]", str(cellValue)) if t.strip()]
    labels = [cleanRoleToken(t) for t in tokens if t.strip().upper() not in primaryRoleTokens]
    return dedupePreserveOrder(labels)

def buildRuleCofactorMap(path):
    ruleMap = {}
    if not os.path.exists(path):
        print(f"  ruleset not found at {path}, using reaction-level cofactors only.")
        return ruleMap
    ruleset = pd.read_csv(path, sep="\t")
    nameCol = "Name" if "Name" in ruleset.columns else ruleset.columns[0]
    hasReact, hasProd = "Reactants" in ruleset.columns, "Products" in ruleset.columns
    for _, row in ruleset.iterrows():
        name = str(row.get(nameCol, "")).strip()
        if not name:
            continue
        entry = {
            "consumed": extractRuleCofactors(row["Reactants"]) if hasReact else [],
            "produced": extractRuleCofactors(row["Products"]) if hasProd else [],
        }
        ruleMap.setdefault(name, entry)
        ruleMap.setdefault(name.split("_")[0], entry)
    return ruleMap

def ruleCofactorsForStep(ruleName, ruleCofactorMap):
    name = str(ruleName).strip()
    entry = ruleCofactorMap.get(name) or ruleCofactorMap.get(name.split("_")[0])
    return (entry["consumed"], entry["produced"]) if entry else ([], [])

def parseReactionSides(reactionSmiles):
    if ">>" not in str(reactionSmiles):
        return [], []
    rStr, pStr = str(reactionSmiles).split(">>", 1)
    return (
        [s.strip() for s in rStr.split(".") if s.strip()],
        [s.strip() for s in pStr.split(".") if s.strip()],
    )

def reactionKey(reactionSmiles):
    """Order- and spacing-independent reaction identity: canonical reactants and products."""
    reactants, products = parseReactionSides(reactionSmiles)
    reactantKey = ".".join(sorted(canonicalizeSmiles(s) for s in reactants))
    productKey  = ".".join(sorted(canonicalizeSmiles(s) for s in products))
    return reactantKey + ">>" + productKey

def buildFeasibilityScoreMap(scoreDF):
    """Map canonical reactionKey -> feasibilityScore_rule3 from reactionDF_DORAXGB_highFeasibility."""
    if feasibilityScoreColumn not in scoreDF.columns:
        raise KeyError(f"Column '{feasibilityScoreColumn}' not in reactionDF_DORAXGB_highFeasibility; available: {list(scoreDF.columns)}")
    if "reactionString" in scoreDF.columns:
        reactionSeries = scoreDF["reactionString"].astype(str)
    elif "reactants" in scoreDF.columns and "products" in scoreDF.columns:
        reactionSeries = scoreDF["reactants"].astype(str) + " >> " + scoreDF["products"].astype(str)
    else:
        raise KeyError("reactionDF_DORAXGB_highFeasibility needs a 'reactionString' column or both 'reactants' and 'products' columns to key reactions.")
    scoreByReaction = {}
    for reactionStr, score in zip(reactionSeries, scoreDF[feasibilityScoreColumn]):
        try:
            scoreValue = float(score)
        except (TypeError, ValueError):
            continue
        scoreByReaction.setdefault(reactionKey(reactionStr), scoreValue)
    return scoreByReaction

def reorderStepDictsForward(stepDicts, canonStarterSet):
    if len(stepDicts) <= 1:
        return stepDicts
    enriched = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants and not products:
            continue
        enriched.append({
            **sd,
            "reactantSet": {canonicalizeSmiles(s) for s in reactants},
            "productSet" : {canonicalizeSmiles(s) for s in products},
        })
    if not enriched:
        return stepDicts
    firstIdx = next((i for i, s in enumerate(enriched) if s["reactantSet"] & canonStarterSet), None)
    if firstIdx is None:
        return stepDicts
    ordered   = [enriched[firstIdx]]
    remaining = [s for i, s in enumerate(enriched) if i != firstIdx]
    while remaining:
        prevProducts = ordered[-1]["productSet"]
        nextIdx = next((i for i, s in enumerate(remaining) if s["reactantSet"] & prevProducts), None)
        if nextIdx is None:
            ordered.extend(remaining)
            break
        ordered.append(remaining[nextIdx])
        remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
    return ordered

def pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants):
    # never let a known cofactor become a backbone node
    pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon and not isKnownCofactor(p)]
    if not pool:
        pool = [p for p in products if canonicalizeSmiles(p) != prevCarrierCanon] or products
    carrier = next((p for p in pool if canonicalizeSmiles(p) == canonTargetSmi), None)
    if carrier is None and nextReactants is not None:
        nextCanon = {canonicalizeSmiles(s) for s in nextReactants}
        feeding = [p for p in pool if canonicalizeSmiles(p) in nextCanon]
        if feeding:
            carrier = max(feeding, key=heavyAtomCount)
    if carrier is None:
        carrier = max(pool, key=heavyAtomCount)
    return carrier

def resolveCofactorLabels(step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap):
    def keepAsCofactor(smi):
        return isKnownCofactor(smi) or heavyAtomCount(smi) <= maxCofactorHeavyAtoms

    consumedSpecies = [s for s in dedupeByCanon([r for r in reactants if canonicalizeSmiles(r) != prevCarrierCanon]) if keepAsCofactor(s)]
    producedSpecies = [s for s in dedupeByCanon([p for p in products  if canonicalizeSmiles(p) != carrierCanon])   if keepAsCofactor(s)]
    consumedLabels = [labelSpecies(s) for s in consumedSpecies]
    producedLabels = [labelSpecies(s) for s in producedSpecies]

    ruleConsumed, ruleProduced = ruleCofactorsForStep(step.get("ruleName", ""), ruleCofactorMap)
    if not consumedLabels and not producedLabels and (ruleConsumed or ruleProduced):
        consumedLabels, producedLabels = ruleConsumed, ruleProduced

    return dedupePreserveOrder(consumedLabels), dedupePreserveOrder(producedLabels)

def buildBackbone(stepDicts, canonStarterSet, canonTargetSmi, ruleCofactorMap):
    steps = []
    for sd in stepDicts:
        reactants, products = parseReactionSides(sd["reactionSmiles"])
        if not reactants or not products:
            continue
        steps.append({**sd, "reactants": reactants, "products": products})
    if not steps:
        return [], []

    firstReactants = steps[0]["reactants"]
    startCarrier = next((s for s in firstReactants if canonicalizeSmiles(s) in canonStarterSet), None)
    if startCarrier is None:
        nonCofactorFirst = [s for s in firstReactants if not isKnownCofactor(s)] or firstReactants
        startCarrier = max(nonCofactorFirst, key=heavyAtomCount)
    nodes            = [startCarrier]
    edges            = []
    prevCarrierCanon = canonicalizeSmiles(startCarrier)

    for stepIdx, step in enumerate(steps):
        reactants, products = step["reactants"], step["products"]
        nextReactants = steps[stepIdx + 1]["reactants"] if stepIdx + 1 < len(steps) else None
        carrier      = pickCarrier(products, prevCarrierCanon, canonTargetSmi, nextReactants)
        carrierCanon = canonicalizeSmiles(carrier)
        consumedLabels, producedLabels = resolveCofactorLabels(
            step, reactants, products, prevCarrierCanon, carrierCanon, ruleCofactorMap
        )
        nodes.append(carrier)
        edges.append({
            "consumed"         : consumedLabels,
            "produced"         : producedLabels,
            "feasibilityScore" : step.get("feasibilityScore"),
        })
        prevCarrierCanon = carrierCanon
    return nodes, edges

def drawMoleculeCairo(smi, sizePx):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None:
        return None
    AllChem.Compute2DCoords(mol)
    drawer = rdMolDraw2D.MolDraw2DCairo(sizePx[0], sizePx[1])
    opts = drawer.drawOptions()
    opts.bondLineWidth   = 3
    opts.padding         = 0.10
    opts.clearBackground = True
    drawer.DrawMolecule(mol)
    drawer.FinishDrawing()
    return Image.open(BytesIO(drawer.GetDrawingText())).convert("RGB")

def textWidth(draw, text, font):
    try:
        bbox = draw.textbbox((0, 0), text, font=font)
        return bbox[2] - bbox[0]
    except Exception:
        return len(text) * 12

def centeredTextX(draw, text, boxWidth, y, font, color="#333333"):
    draw.text(((boxWidth - textWidth(draw, text, font)) // 2, y), text, fill=color, font=font)

def makeCaptionBanner(text, width, height=46):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    centeredTextX(draw, text, width, (height - 34) // 2, captionFont)
    return img

def makeMoleculePanel(smi, caption, imageSizePx):
    molImg = drawMoleculeCairo(smi, imageSizePx)
    if molImg is None:
        return None
    if not caption:
        return molImg
    banner = makeCaptionBanner(caption, molImg.width)
    panel  = Image.new("RGB", (molImg.width, molImg.height + banner.height), "white")
    panel.paste(molImg, (0, 0))
    panel.paste(banner, (0, molImg.height))
    return panel

def joinLabels(labelList):
    return " + ".join([l for l in labelList if l])

def putLabelCentered(draw, text, centerX, y, font, panelWidth, color="#555555"):
    tw = textWidth(draw, text, font)
    x  = int(centerX - tw / 2)
    x  = max(4, min(x, panelWidth - tw - 4))
    draw.text((x, y), text, fill=color, font=font)

def makeHorizontalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cy    = height // 2
    left  = int(width * 0.10)
    right = int(width * 0.90)

    # main straight reaction arrow (left to right)
    draw.line([(left, cy), (right, cy)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.045)
    draw.polygon(
        [(right - int(head * 2.2), cy - head), (right - int(head * 2.2), cy + head), (right, cy)],
        fill=color,
    )

    # feasibility score, centered just below the main arrow, independent of cofactor labels
    if showFeasibilityScore and feasibilityText:
        scoreY = cy + int(0.12 * height)
        putLabelCentered(draw, feasibilityText, (left + right) // 2, scoreY, feasibilityFont, width, color="#1a1a1a")

    if not showCofactorLabels or (not consumedText and not producedText):
        return img

    # curved feeder: reactant enters at the left (tail) end, curve dips down to the
    # main arrow, product leaves at the right (arrowhead) end. Drawn as a quadratic
    # bezier so the endpoints sit under the two labels and the head follows the tangent.
    span  = right - left
    xL    = int(left + 0.15 * span)     # tail end, under the consumed label
    xR    = int(right - 0.05 * span)    # head end, under the produced label
    yTop  = cy - int(0.30 * height)     # both ends sit above the main arrow
    p0    = (xL, yTop)
    ctrl  = ((xL + xR) / 2.0, cy + 0.34 * height)   # control below pulls the curve down to the arrow
    p1    = (xR, yTop)

    thin = max(3, lineWidth // 2)
    pts  = []
    steps = 48
    for i in range(steps + 1):
        t  = i / steps
        mt = 1.0 - t
        x  = mt * mt * p0[0] + 2 * mt * t * ctrl[0] + t * t * p1[0]
        y  = mt * mt * p0[1] + 2 * mt * t * ctrl[1] + t * t * p1[1]
        pts.append((x, y))
    draw.line(pts, fill=color, width=thin, joint="curve")

    # arrowhead at the product (right) end, oriented along the curve tangent ctrl -> p1
    dx, dy = p1[0] - ctrl[0], p1[1] - ctrl[1]
    norm   = math.hypot(dx, dy) or 1.0
    ux, uy = dx / norm, dy / norm
    px, py = -uy, ux
    ah     = int(height * 0.05)
    tip    = p1
    base   = (p1[0] - ux * ah * 1.7, p1[1] - uy * ah * 1.7)
    draw.polygon([
        tip,
        (base[0] + px * ah, base[1] + py * ah),
        (base[0] - px * ah, base[1] - py * ah),
    ], fill=color)

    # labels above each end: consumed at the tail (left), produced at the head (right)
    labelY = yTop - 40
    if consumedText:
        putLabelCentered(draw, consumedText, xL, labelY, cofactorFont, width)
    if producedText:
        putLabelCentered(draw, producedText, xR, labelY, cofactorFont, width)
    return img

def makeVerticalArrow(width, height, consumedText, producedText, feasibilityText="", color=(30, 30, 30), lineWidth=8):
    img  = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    cx     = width // 2
    top    = int(height * 0.14)
    bottom = int(height * 0.86)
    draw.line([(cx, top), (cx, bottom)], fill=color, width=lineWidth)
    head = int(min(width, height) * 0.10)
    draw.polygon(
        [(cx - head, bottom - int(head * 1.6)), (cx + head, bottom - int(head * 1.6)), (cx, bottom)],
        fill=color,
    )
    if showFeasibilityScore and feasibilityText:
        putLabelCentered(draw, feasibilityText, cx, bottom + 6, feasibilityFont, width, color="#1a1a1a")
    midY = (top + bottom) // 2
    if showCofactorLabels and consumedText:
        draw.text((cx + 18, midY - 40), consumedText, fill="#555555", font=cofactorFont)
    if showCofactorLabels and producedText:
        draw.text((cx + 18, midY + 8), producedText, fill="#555555", font=cofactorFont)
    return img

def stackPanelsHorizontally(parts, bgColor="white"):
    valid = [p for p in parts if p is not None]
    if not valid:
        return None
    w = sum(p.width for p in valid)
    h = max(p.height for p in valid)
    canvas = Image.new("RGB", (w, h), bgColor)
    x = 0
    for p in valid:
        canvas.paste(p, (x, 0))
        x += p.width
    return canvas

def stackPanelsVertically(parts, bgColor="white"):
    valid = [p for p in parts if p is not None]
    if not valid:
        return None
    w = max(p.width for p in valid)
    h = sum(p.height for p in valid)
    canvas = Image.new("RGB", (w, h), bgColor)
    y = 0
    for p in valid:
        canvas.paste(p, ((w - p.width) // 2, y))
        y += p.height
    return canvas

def buildCascadeImage(rowData, imageSizePx, arrowSpanPx, direction, ruleCofactorMap):
    canonStarterSet = {canonicalizeSmiles(s) for s in splitStarters(rowData.get("starter_Canonical_SMILES", ""))}
    canonTargetSmi  = canonicalizeSmiles(rowData.get("targetSMILES", ""))
    orderedSteps    = reorderStepDictsForward(rowData["stepDicts"], canonStarterSet)
    nodes, edges    = buildBackbone(orderedSteps, canonStarterSet, canonTargetSmi, ruleCofactorMap)
    if len(nodes) < 2:
        return None

    parts   = []
    lastIdx = len(nodes) - 1
    for idx, smi in enumerate(nodes):
        caption = ""
        if showStarterTargetTags and idx == 0:
            caption = "Starter"
        elif showStarterTargetTags and idx == lastIdx:
            caption = "Target"
        parts.append(makeMoleculePanel(smi, caption, imageSizePx))
        if idx != lastIdx:
            edge            = edges[idx]
            consumedText    = joinLabels(edge["consumed"])
            producedText    = joinLabels(edge["produced"])
            score           = edge.get("feasibilityScore")
            feasibilityText = f"{feasibilityScorePrefix}{score:.3f}" if score is not None else ""
            if direction == "horizontal":
                parts.append(makeHorizontalArrow(arrowSpanPx, imageSizePx[1], consumedText, producedText, feasibilityText))
            else:
                parts.append(makeVerticalArrow(imageSizePx[0], arrowSpanPx, consumedText, producedText, feasibilityText))

    return stackPanelsHorizontally(parts) if direction == "horizontal" else stackPanelsVertically(parts)

def saveImageHighRes(image, basePathNoExt, dpi):
    pngPath = basePathNoExt + ".png"
    pdfPath = basePathNoExt + ".pdf"
    image.save(pngPath, dpi=(dpi, dpi))
    image.save(pdfPath, "PDF", resolution=float(dpi))
    return pngPath, pdfPath

ruleCofactorMap = buildRuleCofactorMap(rulesetPath)


### Render pathways for every starter

In [ ]:
def plotPathwaysForStarter(starterName):
    ctx = starterContexts[starterName]
    if ctx["reactionDF_highFeasible"] is None:
        print(f"[skip] {starterName}: no high-feasibility data (missing {feasibilityFileName})")
        return

    # Per-starter variables the original drivers read.
    reactionDF_highFeasible = ctx["reactionDF_highFeasible"]
    reactionDF_DORAXGB_highFeasibility = ctx["reactionDF_highFeasible"]
    generatedCompoundsDF_highFeasibility = ctx["generatedCompoundsDF_highFeasibility"]
    canonStarterSet = ctx["canonStarterSet"]
    canonHelperSet = ctx["canonHelperSet"]
    doranet_generations = ctx["generations"]
    fileNamePrefix = starterName

    print(f"\n########## {starterName}  (network depth {doranet_generations}) ##########")

    # Feasible-reaction network (all steps here are DORA-XGB high-feasibility by construction)
    feasibleRxnDF = reactionDF_highFeasible.copy()
    if "reactionString" not in feasibleRxnDF.columns:
        feasibleRxnDF["reactionString"] = (
            feasibleRxnDF["reactants"].astype(str) + " >> " + feasibleRxnDF["products"].astype(str)
        )

    # canonical reactant -> feasible reactions consuming it
    reactantToFeasibleRxns = defaultdict(list)
    for idx, row in feasibleRxnDF.iterrows():
        for smi in splitMoleculeString(row["reactants"]):
            reactantToFeasibleRxns[canonicalizeSmiles(smi)].append(idx)

    def carrierProductsOf(row):
        """Non-cofactor, non-helper, non-starter products that can be transformed further."""
        out = []
        for productSmi in splitMoleculeString(row["products"]):
            productCanon = canonicalizeSmiles(productSmi)
            if productCanon in canonHelperSet:
                continue
            if productCanon in canonStarterSet:
                continue
            productMol = Chem.MolFromSmiles(str(productSmi))
            if isCofactorOrEndogenous(productMol, productCanon, cofactorExclusionSet, physicochemicalCutoffs):
                continue
            out.append(productCanon)
        return out

    # Depth-capped BFS over feasible edges: shortest fully-feasible route (<= doranet_generations) to each compound.
    # parent[compound] = (rxnIdx, precursorCanon); absence means "no all-feasible route within doranet_generations".
    parentReaction = {}
    seenCompounds  = set(canonStarterSet)
    searchQueue    = deque((c, 0) for c in canonStarterSet)

    while searchQueue:
        currentCanon, currentDepth = searchQueue.popleft()
        if currentDepth >= doranet_generations:
            continue
        for rxnIdx in reactantToFeasibleRxns.get(currentCanon, []):
            row = feasibleRxnDF.loc[rxnIdx]
            for productCanon in carrierProductsOf(row):
                if productCanon in seenCompounds:
                    continue
                seenCompounds.add(productCanon)
                parentReaction[productCanon] = (rxnIdx, currentCanon)
                searchQueue.append((productCanon, currentDepth + 1))

    def feasibleRouteTo(targetCanon):
        """Reaction strings of the fully-feasible route to a target, forward order; [] if none within doranet_generations."""
        steps = []
        cur   = targetCanon
        while cur in parentReaction:
            rxnIdx, precursorCanon = parentReaction[cur]
            steps.append(feasibleRxnDF.loc[rxnIdx, "reactionString"])
            cur = precursorCanon
        steps.reverse()
        return steps

    def reorderStepsForward(steps, canonStarterSet):
        if len(steps) <= 1:
            return steps
        parsed = []
        for smi in steps:
            if ">>" not in str(smi):
                continue
            rStr, pStr = str(smi).split(">>", 1)
            parsed.append({
                "smi"      : smi,
                "reactants": {canonicalizeSmiles(s.strip()) for s in rStr.split(".") if s.strip()},
                "products" : {canonicalizeSmiles(s.strip()) for s in pStr.split(".") if s.strip()},
            })
        if not parsed:
            return steps
        firstIdx = next((i for i, s in enumerate(parsed) if s["reactants"] & canonStarterSet), None)
        if firstIdx is None:
            return steps
        ordered   = [parsed[firstIdx]]
        remaining = [s for i, s in enumerate(parsed) if i != firstIdx]
        while remaining:
            prevProducts = ordered[-1]["products"]
            nextIdx = next((i for i, s in enumerate(remaining) if s["reactants"] & prevProducts), None)
            if nextIdx is None:
                ordered.extend(remaining)
                break
            ordered.append(remaining[nextIdx])
            remaining = [s for i, s in enumerate(remaining) if i != nextIdx]
        return [s["smi"] for s in ordered]

    def buildRouteImageFromSteps(stepReactions, starterCanon, targetCanon,
                                 headerText, subImageSize, bannerWidth):
        canonStarterSetLocal = {starterCanon}
        numSteps             = len(stepReactions)
        forwardSteps         = reorderStepsForward(stepReactions, canonStarterSetLocal)
        panelBlocks = [makeTextBanner(headerText, bannerWidth=bannerWidth, bannerHeight=44, bgColor="#eef2f7")]
        for stepIdx, rxnSmi in enumerate(forwardSteps, start=1):
            panel = makeCustomReactionPanel(
                rxnSmi, canonStarterSetLocal, targetCanon,
                panelTitle=f"Reaction step {stepIdx} of {numSteps}",
                subImageSize=subImageSize,
            )
            if panel:
                panelBlocks.append(panel)
        return stackPanelsVertically(panelBlocks, gapSize=14, bgColor="white")

    if saveVerticalFigures:
        os.makedirs(figureOutDir, exist_ok=True)

    candidateDF = generatedCompoundsDF_highFeasibility.reset_index(drop=True)

    # Keep only candidates that have a fully-feasible route within doranet_generations
    feasibleRoutes = []
    for _, cand in candidateDF.iterrows():
        targetCanon = canonicalizeSmiles(cand["Canonical_SMILES"])
        steps       = feasibleRouteTo(targetCanon)
        if steps and len(steps) <= doranet_generations:
            firstReactants = str(steps[0]).split(">>", 1)[0]
            starterCanon   = next(
                (canonicalizeSmiles(s) for s in firstReactants.split(".") if canonicalizeSmiles(s) in canonStarterSet),
                cand.get("starter_Canonical_SMILES", ""),
            )
            feasibleRoutes.append((len(steps), targetCanon, starterCanon, steps))

    feasibleRoutes.sort(key=lambda t: t[0])

    print(f"High-feasibility candidates                 : {len(candidateDF):,}")
    print(f"With a fully-feasible route (<= {doranet_generations} steps) : {len(feasibleRoutes):,}")
    stepHistogram = defaultdict(int)
    for nSteps, *_ in feasibleRoutes:
        stepHistogram[nSteps] += 1
    for nSteps in sorted(stepHistogram):
        print(f"  {nSteps}-step feasible pathways : {stepHistogram[nSteps]:,}")

    drawnByStep = defaultdict(int)
    for numSteps, targetCanon, starterCanon, steps in feasibleRoutes:
        if drawnByStep[numSteps] >= numToDisplayPerStep:
            continue
        drawnByStep[numSteps] += 1
        headerText = f"Starter: {str(starterCanon)[:40]}  |  Target: {targetCanon[:50]}  |  Steps: {numSteps}"
        routeImage = buildRouteImageFromSteps(steps, starterCanon, targetCanon, headerText, subImgSize, bannerWidth=1800)
        if routeImage is None:
            continue
        print(f"  [{numSteps}-step #{drawnByStep[numSteps]}] {targetCanon}")
        display(routeImage)
        if saveVerticalFigures:
            basePath         = os.path.join(figureOutDir, f"highFeasPathway_{numSteps}step_{drawnByStep[numSteps]}")
            pngPath, pdfPath = saveImageHighRes(routeImage, basePath, outputDpi)
            print(f"      saved: {pngPath}")

    print("\nDone.")

    scoreByReaction = buildFeasibilityScoreMap(reactionDF_DORAXGB_highFeasibility)

    if saveCascadeFigures:
        os.makedirs(articleFiguresDir, exist_ok=True)

    # Build route-level records directly from the BFS output. feasibleRoutes is a list of
    # (numSteps, targetCanon, starterCanon, steps) where steps is a forward-ordered list of
    # "reactants >> products" strings. Each step's feasibilityScore_rule3 is looked up by a
    # canonical reactionKey so ordering and spacing differences do not break the match.
    routeRecords = []
    for routeIndex, (numSteps, targetCanon, starterCanon, steps) in enumerate(feasibleRoutes):
        stepDicts = [
            {
                "reactionSmiles"   : str(rxnString),
                "ruleName"         : "",
                "feasibilityScore" : scoreByReaction.get(reactionKey(str(rxnString))),
            }
            for rxnString in steps
        ]
        routeRecords.append({
            "routeId"                  : f"bfsRoute_{routeIndex:04d}",
            "numSteps"                 : int(numSteps),
            "starter_Canonical_SMILES" : starterCanon,
            "targetSMILES"             : targetCanon,
            "stepDicts"                : stepDicts,
        })

    routeLevelDF = (
        pd.DataFrame(
            routeRecords,
            columns=["routeId", "numSteps", "starter_Canonical_SMILES", "targetSMILES", "stepDicts"],
        )
        .sort_values(["numSteps", "routeId"])
        .reset_index(drop=True)
    )

    totalSteps        = sum(len(rec["stepDicts"]) for rec in routeRecords)
    missingScoreSteps = sum(1 for rec in routeRecords for sd in rec["stepDicts"] if sd.get("feasibilityScore") is None)
    if missingScoreSteps:
        print(f"Steps with no matched {feasibilityScoreColumn}: {missingScoreSteps:,} of {totalSteps:,}")

    for numSteps, groupDF in routeLevelDF.groupby("numSteps"):
        availableRoutes = groupDF.reset_index(drop=True)
        indices         = selectedRoutes.get(numSteps, list(range(numToDisplay)))
        validIndices    = [i for i in indices if i < len(availableRoutes)]

        print(f"\n{numSteps}-step pathways: {len(availableRoutes):,} available, displaying {len(validIndices):,}")
        if not validIndices:
            print(f"  No valid indices for {numSteps}-step, check selectedRoutes config.")
            continue

        for rank, idx in enumerate(validIndices, start=1):
            rowData      = availableRoutes.loc[idx]
            cascadeImage = buildCascadeImage(rowData, moleculeImageSizePx, arrowSpanPx, layoutDirection, ruleCofactorMap)
            if cascadeImage is None:
                print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']} skipped, no drawable chain.")
                continue
            print(f"  [{rank}/{len(validIndices)}] Route: {rowData['routeId']}")
            display(cascadeImage)
            if saveCascadeFigures:
                basePath         = os.path.join(articleFiguresDir, f"{fileNamePrefix}_pathway_{rowData['routeId']}_{numSteps}step")
                pngPath, pdfPath = saveImageHighRes(cascadeImage, basePath, outputDpi)
                print(f"      saved: {pngPath}")
                print(f"      saved: {pdfPath}")

    print("\nDone.")


# Render pathways for every starter that has high-feasibility data.
for starterName in sorted(starterContexts):
    plotPathwaysForStarter(starterName)
